In [ ]:
!pip install pandas datasets tqdm Pillow

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from typing import Callable
from xml.dom.minidom import Element
from xml.etree import ElementTree as ET

import pandas as pd
from PIL import Image
from datasets import load_dataset, DatasetDict, Dataset
from pandas.core.frame import DataFrame
from tqdm import tqdm

In [ ]:
!mkdir -p dataset/{test,train,validate}

In [ ]:
test_dataset: DataFrame = DataFrame(columns=["image", "text"])
train_dataset: DataFrame = test_dataset.copy()
validate_dataset: DataFrame = train_dataset.copy()

In [ ]:
def _save_OCR_image(args: tuple[Image.Image, str]) -> str:
    image, path = args

    if image.mode != "RGB":
        image = image.convert("RGB")

    image.save(path, format="PNG", compress_level=1)
    return Path(path).name


def export_OCR_split(
        split: Dataset,
        folder: str,
        source: str,
        annotation_column: str = "text",
        extract_text: Callable[[str], str] | None = None,
        max_workers: int = 4,
        batch_size: int = 128
):
    output_dir = Path("dataset") / folder
    output_dir.mkdir(parents=True, exist_ok=True)
    rows: list[dict[str, str]] = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        with tqdm(total=len(split), desc=f"{source} {folder}") as pbar:
            for start in range(0, len(split), batch_size):
                end = min(start + batch_size, len(split))
                batch: dict = split[start:end]

                valid = [
                    (i, text.strip())
                    for i, text in enumerate(executor.map(extract_text, batch[annotation_column])
                                             if extract_text else batch[annotation_column]) if isinstance(text, str) and text.strip()
                ]
                filenames: list[str] = [f"{source}_{folder}_{start + i:06d}.png" for i, _ in valid]
                list(executor.map(_save_OCR_image, zip((batch["image"][i] for i, _ in valid), (str(output_dir / file) for file in filenames))))
                rows.extend({"image": filename, "text": text} for filename, (_, text) in zip(filenames, valid))
                pbar.update(end - start)

    return pd.DataFrame(rows)

# RxHandBD
https://data.mendeley.com/datasets/dsb5r6vskg/2

In [ ]:
!wget https://data.mendeley.com/public-files/datasets/dsb5r6vskg/files/a9179d3a-ee9c-4c1c-bfbd-ac07c6d2b90d/file_downloaded
!unzip file_downloaded -d RxHandBD
!rm -rf file_downloaded

In [ ]:
!mv RxHandBD/Test_Set/* dataset/test
!mv RxHandBD/Train_Set/* dataset/train

In [ ]:
test_dataset = pd.concat(
    [test_dataset, pd.read_csv("RxHandBD/Test_Labels.csv").rename(columns={"Images": "image", "Text": "text"})[["image", "text"]]],
    ignore_index=True
)
train_dataset = pd.concat(
    [train_dataset, pd.read_csv("RxHandBD/Train_Label.csv").rename(columns={"Images": "image", "Text": "text"})[["image", "text"]]],
    ignore_index=True
)

In [ ]:
!rm -rf RxHandBD

# Doctor’s Handwritten Prescription BD dataset
https://www.kaggle.com/datasets/mamun1113/doctors-handwritten-prescription-bd-dataset

In [ ]:
!wget https://www.kaggle.com/api/v1/datasets/download/mamun1113/doctors-handwritten-prescription-bd-dataset
!unzip doctors-handwritten-prescription-bd-dataset
!rm -rf doctors-handwritten-prescription-bd-dataset

In [ ]:
doctor_dataset_dir: Path = Path("Doctor’s Handwritten Prescription BD dataset")
doctor_dataset_files: list[str] = ["Testing/testing_labels.csv", "Training/training_labels.csv", "Validation/validation_labels.csv"]

if not all((doctor_dataset_dir / file).is_file() for file in doctor_dataset_files):
    doctor_dataset_candidates = [
        path for path in Path(".").iterdir() if path.is_dir() and all((path / file).is_file() for file in doctor_dataset_files)
    ]

    if len(doctor_dataset_candidates) != 1:
        raise FileNotFoundError(
            "Could not uniquely locate Doctor's Handwritten Prescription BD dataset. Found: {[str(path) for path in doctor_dataset_candidates]}"
        )

    doctor_dataset_candidates[0].rename(doctor_dataset_dir)


In [ ]:
!mv "Doctor’s Handwritten Prescription BD dataset/Testing/testing_words/"* dataset/test/
!mv "Doctor’s Handwritten Prescription BD dataset/Training/training_words/"* dataset/train/
!mv "Doctor’s Handwritten Prescription BD dataset/Validation/validation_words/"* dataset/validate/

In [ ]:
test_dataset = pd.concat(
    [
        test_dataset,
        pd.read_csv(
            "Doctor’s Handwritten Prescription BD dataset/Testing/testing_labels.csv"
        ).rename(columns={"IMAGE": "image", "MEDICINE_NAME": "text"})[["image", "text"]]
    ],
    ignore_index=True
)
train_dataset = pd.concat(
    [
        train_dataset,
        pd.read_csv(
            "Doctor’s Handwritten Prescription BD dataset/Training/training_labels.csv"
        ).rename(columns={"IMAGE": "image", "MEDICINE_NAME": "text"})[["image", "text"]]
    ],
    ignore_index=True
)
validate_dataset = pd.concat(
    [
        validate_dataset,
        pd.read_csv(
            "Doctor’s Handwritten Prescription BD dataset/Validation/validation_labels.csv"
        ).rename(columns={"IMAGE": "image", "MEDICINE_NAME": "text"})[["image", "text"]]
    ],
    ignore_index=True
)

In [ ]:
!rm -rf "Doctor’s Handwritten Prescription BD dataset"

# IAM-line
https://huggingface.co/datasets/Teklia/IAM-line

In [ ]:
IAM_dataset: DatasetDict = load_dataset("Teklia/IAM-line")

In [ ]:
train_dataset = pd.concat([train_dataset, export_OCR_split(IAM_dataset["train"], "train", "IAM")], ignore_index=True)
test_dataset = pd.concat([test_dataset, export_OCR_split(IAM_dataset["test"], "test", "IAM")], ignore_index=True)
validate_dataset = pd.concat([validate_dataset, export_OCR_split(IAM_dataset["validation"], "validate", "IAM")], ignore_index=True)

In [ ]:
test_dataset.to_csv("dataset/test.csv", index=False)
train_dataset.to_csv("dataset/train.csv", index=False)
validate_dataset.to_csv("dataset/validate.csv", index=False)